# BRACED — Fresh End-to-End Training Notebook

Built from scratch. Fixes the `-0.8` reward plateau (seed was wrong + structure
scored 1-hop). Pipeline:

1. **Config + load** DedupKG (type-aware, deduplicated) + fast entity matcher.
2. **Load BioHopR**; correct field mapping (`hop2`=SOURCE, `hop1`=BRIDGE, `answer`=GOLD).
3. **SFT warmup** — distill 2-sub-query plans, LoRA-train so GRPO has a warm start
   (avoids cold-start where empty retrieval => no reward signal).
4. **Smart Environment** — 2-hop typed retrieval; gives `retrieved_hop1` + `retrieved_hop2`.
5. **Reward** — `R = 0.1*format + 0.4*structure(2-tier) + 0.5*outcome(proxy F1)`.
   Proxy F1 over the retrieved set at TRAIN (free, no LLM). 
6. **GRPO** — hand-rolled (no trl), group-relative advantage, separate
   generate/update phases (25x speedup), checkpoint/100, early stop, cap 1500.
7. **Eval** — 764 test, GPT-4o-mini **Strict Formatter** for TRUE F1.

> Train reward = KG-grounded proxy (no LLM-in-the-loop). Eval = Strict Formatter.
> State this in the paper: no LLM-judge leakage during training.


## 1. Config + dependencies

In [ ]:
# Run once. PyTorch 2.6+ pod recommended.
!pip install -q bitsandbytes peft accelerate transformers datasets pyahocorasick openai
import os, sys, json, re, ast, time, random, math, pickle
from pathlib import Path
import numpy as np
import torch

sys.path.insert(0, '/workspace')
random.seed(0); np.random.seed(0); torch.manual_seed(0)

CFG = dict(
    BACKBONE   = 'Qwen/Qwen2.5-7B-Instruct',
    LOOKUPS    = '/workspace/data/processed/entity_lookups.pkl',
    RELATIONS  = '/workspace/data/processed/relation_index.pkl',
    SPLITS     = '/workspace/cache/biohopr_splits.pkl',
    OUT        = '/workspace/outputs/rl_dqd_fresh',
    TOP_K      = 25,
    # SFT
    SFT_N      = 2000,      # distilled examples to build (cap for time)
    SFT_EPOCHS = 1,
    # GRPO  (MAX_STEPS now counts REAL optimizer updates; skipped no-signal groups do NOT count)
    GROUP_G      = 8,       # rollouts per question
    MAX_STEPS    = 1500,    # number of gradient UPDATES to perform
    MAX_ROLLOUTS = 8000,    # hard safety cap on questions sampled (stops runaway if skip-rate is high)
    CKPT_EVERY   = 100,     # checkpoint + DEV eval every N updates
    LR           = 1e-5,
    STD_NORM     = False,   # False = Dr.GRPO style (no per-group std norm); True = classic GRPO
    KL_BETA      = 0.0,     # reserved: reference-model KL is NOT wired in this REINFORCE-with-baseline variant
    TEMP         = 1.3,
    MAX_NEW      = 160,
    EARLY_PATIENCE = 5,     # stop if DEV F1 doesn't improve for N checkpoints
    DEV_N        = 200,     # held-out dev examples (carved from TRAIN) for selection; TEST stays untouched
    # reward weights
    W_FORMAT=0.1, W_STRUCT=0.4, W_OUTCOME=0.5,
    W_SQ1=0.4, W_SQ2=0.6,
    # eval
    EVAL_N = 764,
)
ANSWER_TYPES = ['disease','drug','gene/protein','phenotype','effect/phenotype']
Path(CFG['OUT']).mkdir(parents=True, exist_ok=True)
print('Config ready. CUDA:', torch.cuda.is_available())


## 2. Load DedupKG + fast entity matcher (Aho-Corasick)

In [ ]:
from primekg_dedup import DedupKG
kg = DedupKG.load(CFG['LOOKUPS'], CFG['RELATIONS'], verbose=True)

# Fast multi-pattern entity matcher so we DON'T loop 129K names per sub-query.
try:
    import ahocorasick
    A = ahocorasick.Automaton()
    for name, eid in kg.name_to_id.items():
        if len(name) >= 4:
            A.add_word(name.lower(), (name.lower(), eid))
    A.make_automaton()
    HAVE_AC = True
    print('Aho-Corasick matcher built:', len(kg.name_to_id), 'names')
except Exception as e:
    HAVE_AC = False
    print('ahocorasick unavailable, falling back to slow match:', e)

def extract_entities(text, max_e=3):
    """Return up to max_e canonical entity ids mentioned in text."""
    t = ' ' + text.lower() + ' '
    hits = {}
    if HAVE_AC:
        for end, (name, eid) in A.iter(t):
            start = end - len(name) + 1
            # word-ish boundary check to avoid substring noise
            if (t[start-1] in ' (),.;:') and (t[end+1] in ' (),.;:'):
                can = kg.get_canonical_id(eid)
                hits[can] = max(hits.get(can, 0), len(name))
    else:
        for name, eid in kg.name_to_id.items():
            if len(name) >= 4 and (' '+name+' ') in t:
                can = kg.get_canonical_id(eid); hits[can] = max(hits.get(can,0), len(name))
    # longest matches first
    return [e for e,_ in sorted(hits.items(), key=lambda kv: -kv[1])][:max_e]

print('entity extractor ready')

## 3. Load BioHopR (correct field mapping)

In [ ]:
splits = pickle.load(open(CFG['SPLITS'],'rb'))
train_set, test_set = splits['train'], splits['test']

def parse_gold(raw):
    if isinstance(raw, list): return [str(x).strip() for x in raw]
    try: return [str(x).strip() for x in ast.literal_eval(str(raw).replace('array(','').rstrip(')'))]
    except: return [str(raw).strip()]

# Field semantics (verified):
#   hop2  = SOURCE entity (start of reasoning)
#   hop1  = BRIDGE entity
#   answer = GOLD targets ; target_type = answer type ; hop1_type = bridge type
def pack(s):
    return dict(
        question = s.get('hop2_question', s.get('prompt','')),
        source   = str(s.get('hop2','')),
        bridge   = str(s.get('hop1','')),
        bridge_type = str(s.get('hop1_type','')).strip(),
        target_type = str(s.get('target_type','')).strip(),
        gold     = parse_gold(s.get('answer')),
    )

_all_train = [pack(s) for s in train_set]
test       = [pack(s) for s in test_set]      # reserved for FINAL eval only (no leakage)

# ---- hold out a DEV split from TRAIN for checkpoint selection / early stopping ----
_rng = random.Random(0)
_idx = list(range(len(_all_train))); _rng.shuffle(_idx)
_n_dev = min(CFG['DEV_N'], max(1, len(_all_train)//10))
_dev_idx = set(_idx[:_n_dev])
train = [e for i,e in enumerate(_all_train) if i not in _dev_idx]   # GRPO trains on this
dev   = [_all_train[i] for i in _idx[:_n_dev]]                       # selection/early-stop on this
print('train', len(train), 'dev', len(dev), 'test', len(test), '(dev is held out from train; test untouched)')
print('example:', {k:(v[:60] if isinstance(v,str) else v) for k,v in train[0].items()})


## 4. Smart Environment — 2-hop typed retrieval

In [ ]:
def typed_neighbors(eid, ttype, cap=None):
    ns = kg.get_neighbors(eid, top_k=None, target_type=(ttype or None))
    ns = list(ns)
    return ns[:cap] if cap else ns

def run_two_hop(seed_ids, bridge_type, target_type, top_k):
    """hop1: seed -> bridge-typed ; hop2: bridge -> answer-typed.
    Returns (hop1_ids, hop2_top, hop2_full)."""
    hop1 = set()
    for sid in seed_ids:
        hop1 |= set(typed_neighbors(sid, bridge_type))
    hop2 = set()
    for b in list(hop1)[:300]:           # cap bridge fan-out for speed
        hop2 |= set(typed_neighbors(b, target_type))
    hop2_top = list(hop2)[:top_k]
    return hop1, set(hop2_top), hop2

USE_SOURCE_FALLBACK = False   # no gold-source leakage (set True only for warm-start ablation)

def env_execute(sub_queries, ex):
    """Decoupled: env extracts entities from the policy's OWN NL sub-queries.
    No gold-source leakage. If the policy emits no usable sub-query, retrieval
    is empty and the reward reflects that (this is the learning signal)."""
    sq1 = sub_queries[0] if sub_queries else ''
    seeds = extract_entities(sq1)
    if not seeds and USE_SOURCE_FALLBACK:
        sid = kg.lookup(ex['source'])
        if sid: seeds = [sid]
    hop1, hop2_top, hop2_full = run_two_hop(seeds, ex['bridge_type'],
                                            ex['target_type'], CFG['TOP_K'])
    pred_names = [kg.id_to_name.get(x,'?') for x in hop2_top]
    return dict(seeds=seeds, hop1=hop1, hop2_top=hop2_top,
                hop2_full=hop2_full, pred_names=pred_names)

print('Smart Environment ready')

## 5. Reward — format + 2-tier structure + proxy-F1 outcome

In [ ]:
def resolve_ids(names):
    out=set()
    for n in names:
        i=kg.lookup(n)
        if i: out.add(i)
    return out

def r_format(sqs):
    if not sqs: return -1.0
    if not (1 <= len(sqs) <= 3): return -0.5
    ok = sum(1 for s in sqs if isinstance(s,str) and len(s.split())>=3)
    return ok/len(sqs)

def r_structure(hop1_ids, hop2_ids, bridge_name, gold_names):
    bid = kg.lookup(bridge_name) if bridge_name else None
    sq1 = 1.0 if (bid and bid in hop1_ids) else 0.0
    gids = resolve_ids(gold_names)
    sq2 = (len(hop2_ids & gids)/len(gids)) if gids else 0.0
    return CFG['W_SQ1']*sq1 + CFG['W_SQ2']*sq2, sq1, sq2

def r_outcome_f1(pred_names, gold_names):
    P={p.lower().strip() for p in pred_names if p and p.strip()}
    G={g.lower().strip() for g in gold_names}
    if not P and not G: return 1.0
    if not P or not G:  return 0.0
    tp=len(P&G); fp=len(P-G); fn=len(G-P)
    pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
    return 2*pr*rc/(pr+rc) if pr+rc else 0.0

def compute_reward(ex, sqs, envout):
    rf = r_format(sqs)
    rs, sq1, sq2 = r_structure(envout['hop1'], envout['hop2_full'],
                               ex['bridge'], ex['gold'])
    # proxy F1 over capped retrieved set (no LLM at train time)
    ro = r_outcome_f1(envout['pred_names'], ex['gold'])
    total = CFG['W_FORMAT']*rf + CFG['W_STRUCT']*rs + CFG['W_OUTCOME']*ro
    return dict(total=total, fmt=rf, struct=rs, sq1=sq1, sq2=sq2, outcome=ro)
print('reward ready')

## 6. Load policy (Qwen2.5-7B + QLoRA 4-bit)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(CFG['BACKBONE'])
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = 'left'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(CFG['BACKBONE'], quantization_config=bnb,
                                            device_map='auto', torch_dtype=torch.bfloat16)
base = prepare_model_for_kbit_training(base)
base.enable_input_require_grads()  # fix: grads flow through checkpointing
# lora_dropout 0.0 (was 0.05): removes train/sample dropout mismatch in policy-gradient
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias='none',
                  task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj'])
model = get_peft_model(base, lora)
model.print_trainable_parameters()

SYS = "You are a biomedical reasoning planner. Decompose the question into 1-2 "\
      "natural-language sub-queries that retrieve evidence from a knowledge graph. "\
      "Output ONLY JSON: [{\"sub_query\": \"...\"}]."

def build_prompt(question):
    msgs = [{'role':'system','content':SYS},
            {'role':'user','content':f'Question: {question}\nJSON:'}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse_subqueries(text):
    try:
        seg = text[text.find('['):text.rfind(']')+1]
        arr = json.loads(seg)
        sqs = [d.get('sub_query','').strip() for d in arr if isinstance(d,dict)]
        return [s for s in sqs if s][:3]
    except Exception:
        return []
print('policy ready')

## 7. SFT warmup
Build distilled 2-sub-query targets from the gold reasoning chain (source -> bridge
-> answer) so the policy starts able to emit entity-bearing sub-queries.
We construct targets programmatically from `source`/`bridge` (no API needed).

In [ ]:
from torch.utils.data import DataLoader
import random as _rnd

# Template VARIATIONS so the model learns the concept, not one exact string.
_SQ1_TPL = [
    "Find the {bt} associated with {src}.",
    "Identify a {bt} linked to {src}.",
    "Which {bt} is related to {src}?",
    "Retrieve the {bt} connected to {src}.",
    "List the {bt} associated with {src}.",
]
_SQ2_TPL = [
    "Find the {tt} related to that {bt}.",
    "Identify the {tt} associated with the {bt}.",
    "Which {tt} corresponds to that {bt}?",
    "Retrieve the {tt} linked to the {bt}.",
    "List the {tt} for that {bt}.",
]

def make_sft_target(ex):
    sq1 = _rnd.choice(_SQ1_TPL).format(bt=ex['bridge_type'], src=ex['source'])
    sq2 = _rnd.choice(_SQ2_TPL).format(tt=ex['target_type'], bt=ex['bridge_type'])
    return json.dumps([{"sub_query": sq1}, {"sub_query": sq2}], ensure_ascii=False)

def build_sft_examples(data, n):
    _rnd.seed(0)
    out=[]
    for ex in data[:n]:
        if not ex['source'] or not ex['bridge']: continue
        out.append((build_prompt(ex['question']), make_sft_target(ex)))
    return out

sft_ex = build_sft_examples(train, CFG['SFT_N'])
print('SFT examples:', len(sft_ex), '(with 5x5=25 template combos)')

def collate(batch):
    tok.padding_side = 'right'
    input_ids_list, labels_list = [], []
    for p, t in batch:
        pid = tok(p, add_special_tokens=False)['input_ids']
        tid = tok(t + tok.eos_token, add_special_tokens=False)['input_ids']
        ids = pid + tid
        lab = [-100]*len(pid) + tid[:]
        input_ids_list.append(ids); labels_list.append(lab)
    maxlen = min(max(len(x) for x in input_ids_list), 512)
    input_ids, labels, attn = [], [], []
    for ids, lab in zip(input_ids_list, labels_list):
        ids, lab = ids[:maxlen], lab[:maxlen]
        pad = maxlen - len(ids)
        input_ids.append(ids + [tok.pad_token_id]*pad)
        labels.append(lab + [-100]*pad)
        attn.append([1]*len(ids) + [0]*pad)
    import torch as _t
    return {'input_ids':_t.tensor(input_ids),'attention_mask':_t.tensor(attn),
            'labels':_t.tensor(labels)}

_b = collate(sft_ex[:1])
_lab = _b['labels'][0]
_kept = tok.decode([i for i,l in zip(_b['input_ids'][0].tolist(), _lab.tolist()) if l!=-100])
print('--- SANITY: text the model is trained to PRODUCE (unmasked part) ---')
print(repr(_kept))

# FEWER steps, lower LR, early stop -> avoid memorization plateau
model.config.use_cache = False
model.gradient_checkpointing_enable()
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=3e-5, weight_decay=0.01)
from transformers import get_cosine_schedule_with_warmup
MAX_SFT_STEPS = 200
dl = DataLoader(sft_ex, batch_size=4, shuffle=True, collate_fn=collate)
sched = get_cosine_schedule_with_warmup(opt, int(0.05*MAX_SFT_STEPS), MAX_SFT_STEPS)

model.train()
done=False
for epoch in range(CFG['SFT_EPOCHS']):
    if done: break
    for step, enc in enumerate(dl):
        enc = {k:v.to(model.device) for k,v in enc.items()}
        loss = model(**enc).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step(); sched.step(); opt.zero_grad()
        gstep = epoch*len(dl)+step
        if gstep % 20 == 0: print(f"  SFT step {gstep} loss={loss.item():.3f}")
        # early-stop BEFORE memorization (keep some entropy for GRPO exploration)
        if loss.item() < 0.3 and gstep >= 60:
            print(f"  early-stop at step {gstep} (loss={loss.item():.3f} < 0.3 -> keep exploration entropy)")
            done=True; break
        if gstep >= MAX_SFT_STEPS:
            done=True; break
model.save_pretrained(CFG['OUT']+'/sft_adapter')
print('✓ SFT done')

# POST-SFT check: greedy AND sampled diversity
model.config.use_cache = True
model.eval()
_p = build_prompt(train[0]['question'])
_e = tok(_p, return_tensors='pt').to(model.device)
with torch.no_grad():
    _o = model.generate(**_e, do_sample=False, max_new_tokens=120, pad_token_id=tok.pad_token_id)
print('--- POST-SFT greedy ---')
print(repr(tok.decode(_o[0][_e["input_ids"].shape[1]:], skip_special_tokens=True)))

# diversity probe: 4 samples at T=1.3
with torch.no_grad():
    _o2 = model.generate(**_e, do_sample=True, temperature=1.3, top_p=0.95,
                         num_return_sequences=4, max_new_tokens=120,
                         pad_token_id=tok.pad_token_id)
print('--- POST-SFT diversity probe (4 samples @ T=1.3) ---')
_uniq=set()
for k in range(4):
    s = tok.decode(_o2[k][_e["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"  s{k}: {repr(s)[:160]}")
    _uniq.add(s)
print(f"  unique samples: {len(_uniq)}/4  (need >1 for GRPO variance)")
model.train()

## 8. Rollout + sequence log-prob (separate generate / update)

In [ ]:
@torch.no_grad()
def sample_completions(prompt, g, temp, max_new):
    """Generate g completions for one prompt. Returns list of (text, comp_ids)."""
    model.eval()                                           # FIX: kill dropout during sampling
    model.config.use_cache = True
    enc = tok(prompt, return_tensors='pt').to(model.device)
    plen = enc['input_ids'].shape[1]
    out = model.generate(**enc, do_sample=True, temperature=temp, top_p=0.95,
                         num_return_sequences=g, max_new_tokens=max_new,
                         pad_token_id=tok.pad_token_id)
    res=[]
    for seq in out:
        comp_ids = seq[plen:]
        text = tok.decode(comp_ids, skip_special_tokens=True)
        res.append((text, comp_ids))
    return res, plen

def seq_logprob(prompt, comp_ids):
    """Recompute log-prob of comp_ids under current policy (WITH grad)."""
    model.train()                                          # back to train for grad
    model.config.use_cache = False
    enc = tok(prompt, return_tensors='pt').to(model.device)
    pids = enc['input_ids'][0]
    full = torch.cat([pids, comp_ids.to(model.device)]).unsqueeze(0)
    attn = torch.ones_like(full)
    out = model(input_ids=full, attention_mask=attn)
    logits = out.logits[0, :-1]
    targets = full[0,1:]
    logp = torch.log_softmax(logits, dim=-1)
    tok_lp = logp[range(len(targets)), targets]
    comp_lp = tok_lp[len(pids)-1:]
    return comp_lp.sum()
print('rollout utils ready')

## 9. GRPO training loop (hand-rolled)

In [ ]:
def grpo_train(train_data, dev_data, eval_fn):
    model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG['LR'])
    best_f1, patience, history = -1.0, 0, []
    updates = 0     # REAL optimizer steps (skipped groups don't count)
    rollouts = 0    # total questions sampled
    skipped = 0     # groups with no reward variance (no gradient signal)
    order = list(range(len(train_data))); random.shuffle(order); ptr = 0

    while updates < CFG['MAX_STEPS'] and rollouts < CFG['MAX_ROLLOUTS']:
        if ptr >= len(order): random.shuffle(order); ptr = 0
        ex = train_data[order[ptr]]; ptr += 1
        rollouts += 1
        prompt = build_prompt(ex['question'])

        # ---- generate phase (no grad) ----
        comps, _ = sample_completions(prompt, CFG['GROUP_G'], CFG['TEMP'], CFG['MAX_NEW'])
        rewards = []; metas = []
        for text, comp_ids in comps:
            sqs = parse_subqueries(text)
            envout = env_execute(sqs, ex)
            r = compute_reward(ex, sqs, envout)
            rewards.append(r['total']); metas.append((comp_ids, r))
        rewards = np.array(rewards, dtype=np.float32)

        # ---- skip groups with no learning signal (all rollouts identical reward) ----
        if float(rewards.std()) < 1e-6:
            skipped += 1
            continue                       # no update; checkpoint schedule is keyed on `updates`, so it is NOT starved

        # ---- group-relative advantage ----
        adv = rewards - rewards.mean()
        if CFG['STD_NORM']:
            adv = adv / (rewards.std() + 1e-6)

        # ---- update phase: per-rollout backward = low memory (no giant summed graph) ----
        opt.zero_grad()
        for (comp_ids, r), a in zip(metas, adv):
            lp = seq_logprob(prompt, comp_ids)
            (-(float(a) / CFG['GROUP_G']) * lp).backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step()
        updates += 1

        if updates % 20 == 0:
            om = float(np.mean([m[1]['outcome'] for m in metas]))
            print(f"  upd {updates} | rollouts {rollouts} skip {skipped/max(rollouts,1):.0%} "
                  f"| reward {rewards.mean():.3f}±{rewards.std():.3f} outcomeF1 {om:.3f}")

        # ---- checkpoint + DEV eval (fires reliably every CKPT_EVERY *updates*) ----
        if updates % CFG['CKPT_EVERY'] == 0:
            model.save_pretrained(f"{CFG['OUT']}/ckpt_{updates}")
            f1 = eval_fn(dev_data, n=CFG['DEV_N'])          # DEV, never test
            history.append((updates, float(rewards.mean()), f1))
            print(f"  [ckpt {updates}] DEV proxy-F1={f1:.4f} (best {best_f1:.4f}) "
                  f"| skip_rate={skipped/max(rollouts,1):.0%}")
            if f1 > best_f1 + 1e-4:
                best_f1 = f1; patience = 0
                model.save_pretrained(f"{CFG['OUT']}/best")
                print(f"    -> new best, saved to {CFG['OUT']}/best")
            else:
                patience += 1
                if patience >= CFG['EARLY_PATIENCE']:
                    print(f"  early stop at update {updates} (no DEV gain x{patience})")
                    break

    skip_rate = skipped/max(rollouts,1)
    print(f"training done. updates={updates}, rollouts={rollouts}, "
          f"skipped={skipped} ({skip_rate:.0%}), best DEV-F1={best_f1:.4f}")
    if skip_rate > 0.5:
        print("  WARNING: >50% of groups had no reward variance (wasted compute). "
              "Consider lowering TEMP, or inspecting format/parse so rollouts differ.")
    json.dump([{'step':s,'rw':float(r),'f1':float(f)} for s,r,f in history],
              open(CFG['OUT']+'/history.json','w'), indent=2)
    return history
print('GRPO loop ready (dev-based selection, reliable checkpoints, skip-rate logging, low-mem backward)')


## 10. Quick proxy-F1 eval (used during training, no API)

In [ ]:
@torch.no_grad()
def quick_eval(data, n=200):
    model.eval()
    f1s=[]
    for ex in data[:n]:
        prompt = build_prompt(ex['question'])
        enc = tok(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**enc, do_sample=False, max_new_tokens=CFG['MAX_NEW'],
                             pad_token_id=tok.pad_token_id)
        text = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        sqs = parse_subqueries(text)
        envout = env_execute(sqs, ex)
        f1s.append(r_outcome_f1(envout['pred_names'], ex['gold']))
    model.train()
    return float(np.mean(f1s)) if f1s else 0.0
print('quick_eval ready (pass a dataset: use `dev` during training; use `test` only for the final number)')


## 10b. Smoke test — RUN THIS BEFORE full GRPO
Generates 4 rollouts for one question and prints sub-queries + reward breakdown.
PASS criteria: rewards DIFFER across the 4 rollouts (variance > 0) and at least
one rollout has sq2 > 0 (gold reachable). If all rewards are identical/negative,
stop and debug before wasting GPU on full training.

In [ ]:
# Smoke test before committing to 1500 steps of GRPO.
ex = train[0]
print("QUESTION:", ex['question'])
print("source(hop2):", ex['source'], "| bridge(hop1):", ex['bridge'],
      "| target_type:", ex['target_type'], "| n_gold:", len(ex['gold']))
print("-"*70)

prompt = build_prompt(ex['question'])
comps, _ = sample_completions(prompt, 4, CFG['TEMP'], CFG['MAX_NEW'])
totals = []
for j, (text, ids) in enumerate(comps):
    sqs = parse_subqueries(text)
    envout = env_execute(sqs, ex)
    r = compute_reward(ex, sqs, envout)
    totals.append(r['total'])
    print(f"rollout {j}: n_sqs={len(sqs)} seeds={len(envout['seeds'])} "
          f"pred={len(envout['pred_names'])} | reward={r['total']:.3f} "
          f"(fmt={r['fmt']:.2f} sq1={r['sq1']:.0f} sq2={r['sq2']:.2f} out={r['outcome']:.2f})")
    print(f"   RAW: {repr(text)[:200]}")
    print(f"   SQ:  {sqs}")

import numpy as np
std = float(np.std(totals))
print("-"*70)
print(f"reward std across 4 rollouts = {std:.4f}")
if std > 1e-6:
    print("✓ PASS — reward varies, GRPO has gradient. Safe to run full training.")
else:
    print("✗ FAIL — rewards identical. Inspect RAW above:")
    print("  - empty raw -> generation broken (eos immediately?)")
    print("  - garbled raw -> SFT overfit, try CFG['TEMP']=0.7")
    print("  - JSON raw but parse=[] -> parse_subqueries bug")


## 11. Run GRPO

In [ ]:
history = grpo_train(train, dev, quick_eval)
if history:
    best = max(history, key=lambda h: h[2])
    print('best DEV checkpoint:', {'update': best[0], 'dev_F1': round(best[2], 4)})
    print('--> load the adapter at', CFG['OUT'] + '/best', 'and run the FINAL TEST eval cells below (no leakage).')
else:
    print('No updates were made (reward had no variance in every sampled group). '
          'Lower TEMP or inspect reward/format parsing before retraining.')


## 12. Final eval — GPT-4o-mini Strict Formatter (TRUE F1)
This is the number reported in the paper. API called once per test question (~764).

In [ ]:
"""Paste as a NEW cell BEFORE Final Eval (## 12).

Loads the best LoRA adapter from training, fixes the OpenAI key if needed,
and runs the full Strict Formatter eval on all 764 test questions."""
import os, glob, json

# --- 1. confirm where the best checkpoint is ---
best_dir = CFG['OUT'] + '/best'
ckpt_dirs = sorted(glob.glob(CFG['OUT'] + '/ckpt_*'),
                   key=lambda p: int(p.split('_')[-1]))
print("checkpoints saved:")
for d in ckpt_dirs: print(" ", d)
print("best dir exists:", os.path.isdir(best_dir), "->", best_dir)

# --- 2. load best (or fall back to step 1300 manually if /best is missing) ---
from peft import PeftModel
target = best_dir if os.path.isdir(best_dir) else f"{CFG['OUT']}/ckpt_1300"
print(f"\nLoading adapter: {target}")
# unload current adapter, load best
try:
    model.unload()
except Exception:
    pass
model.load_adapter(target, adapter_name='best')
model.set_adapter('best')
model.eval()
print("✓ best adapter active")

# --- 3. quick sanity: greedy on first 3 test questions ---
import torch
for i in range(3):
    prompt = build_prompt(test[i]['question'])
    enc = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, do_sample=False, max_new_tokens=CFG['MAX_NEW'],
                             pad_token_id=tok.pad_token_id)
    txt = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    sqs = parse_subqueries(txt)
    envout = env_execute(sqs, test[i])
    proxy = r_outcome_f1(envout['pred_names'], test[i]['gold'])
    print(f"  test {i}: n_sqs={len(sqs)} pred={len(envout['pred_names'])} "
          f"proxy_F1={proxy:.3f}")

# --- 4. OpenAI key ---
if not os.environ.get('OPENAI_API_KEY'):
    from getpass import getpass
    os.environ['OPENAI_API_KEY'] = getpass('OpenAI key (regenerated): ')
# smoke-test the key with a 1-token call
from openai import OpenAI
oai = OpenAI()
_t = oai.chat.completions.create(model='gpt-4o-mini', max_tokens=1,
        messages=[{'role':'user','content':'hi'}])
print("✓ OpenAI key works")

print("\nReady. Now run the Final Eval cell (## 12).")


In [ ]:
from openai import OpenAI
if not os.environ.get('OPENAI_API_KEY'):
    from getpass import getpass; os.environ['OPENAI_API_KEY']=getpass('OpenAI key: ')
oai = OpenAI()

STRICT = ("You are a strict formatter. From the candidate entities, output ONLY those "
          "that answer the question, as a comma-separated list. Use ONLY the candidates; "
          "do NOT add knowledge. If none apply, output NONE.\n"
          "Question: {q}\nCandidates: {c}\nAnswer:")

def strict_format(q, cand_names):
    if not cand_names: return []
    r = oai.chat.completions.create(model='gpt-4o-mini', temperature=0, max_tokens=300,
        messages=[{'role':'user','content':STRICT.format(q=q, c=', '.join(cand_names))}])
    txt = r.choices[0].message.content.strip()
    if txt.upper()=='NONE': return []
    return [x.strip() for x in txt.split(',') if x.strip()]

@torch.no_grad()
def final_eval(data, use_best=True):
    if use_best:
        from peft import PeftModel
        print('(load best adapter manually if needed:', CFG['OUT']+'/best )')
    model.eval()
    tp=fp=fn=0; rows=[]
    for i,ex in enumerate(data[:CFG['EVAL_N']]):
        prompt = build_prompt(ex['question'])
        enc = tok(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**enc, do_sample=False, max_new_tokens=CFG['MAX_NEW'],
                             pad_token_id=tok.pad_token_id)
        text = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        sqs = parse_subqueries(text)
        envout = env_execute(sqs, ex)
        pred = strict_format(ex['question'], envout['pred_names'])
        P={p.lower() for p in pred}; G={g.lower() for g in ex['gold']}
        tp+=len(P&G); fp+=len(P-G); fn+=len(G-P)
        rows.append(dict(q=ex['question'], pred=pred, gold=ex['gold']))
        if (i+1)%100==0: print(f"  eval {i+1}/{CFG['EVAL_N']}")
    pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
    micro=2*pr*rc/(pr+rc) if pr+rc else 0
    json.dump(dict(micro_f1=micro,precision=pr,recall=rc,n=len(rows)),
              open(CFG['OUT']+'/final_eval.json','w'), indent=2)
    json.dump(rows, open(CFG['OUT']+'/final_preds.json','w'), indent=2)
    print(f"\n=== BRACED (B11) FINAL: micro-F1={micro:.4f} (P={pr:.3f} R={rc:.3f}) ===")
    return micro

final_eval(test)   # <- run after training (and after loading /best)

In [ ]:
"""Re-eval with an INCLUSIVE Strict Formatter prompt.
Symptom: P=0.819, R=0.029 -> SF was too conservative (output 1 of 4-68 golds).
Fix: prompt SF to list ALL applicable candidates, regardless of singular phrasing."""
import os, json
from openai import OpenAI
oai = OpenAI()

# INCLUSIVE prompt: explicitly tell SF to output ALL applicable, ignore singular phrasing
STRICT_INCLUSIVE = (
    "You are an evidence-based formatter. From the candidate entities below, "
    "output EVERY entity that correctly answers the question. "
    "The question may use singular phrasing (e.g., 'Name a drug') but you must "
    "list ALL applicable entities from the candidates. "
    "Use ONLY the provided candidates; do not add outside knowledge. "
    "Output format: comma-separated list of entity names. "
    "If none apply, output NONE.\n"
    "Question: {q}\nCandidates: {c}\nAll applicable answers:"
)

def strict_format_inclusive(q, cand_names):
    if not cand_names: return []
    r = oai.chat.completions.create(model='gpt-4o-mini', temperature=0, max_tokens=600,
        messages=[{'role':'user','content':STRICT_INCLUSIVE.format(q=q, c=', '.join(cand_names))}])
    txt = r.choices[0].message.content.strip()
    if txt.upper().startswith('NONE'): return []
    return [x.strip() for x in txt.split(',') if x.strip()]

# Diagnostic on 10 test before full re-eval
print("="*70)
print("Diagnostic: compare conservative vs inclusive SF on 10 test")
print("="*70)
import torch
model.eval()
for i in range(10):
    ex = test[i]
    prompt = build_prompt(ex['question'])
    enc = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, do_sample=False, max_new_tokens=CFG['MAX_NEW'],
                             pad_token_id=tok.pad_token_id)
    txt = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    sqs = parse_subqueries(txt)
    envout = env_execute(sqs, ex)
    pred = strict_format_inclusive(ex['question'], envout['pred_names'])
    P={p.lower() for p in pred}; G={g.lower() for g in ex['gold']}
    tp=len(P&G); fp=len(P-G); fn=len(G-P)
    pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
    f1=2*pr*rc/(pr+rc) if pr+rc else 0
    print(f"  t{i}: cand={len(envout['pred_names'])} pred={len(pred)} gold={len(ex['gold'])} "
          f"P={pr:.2f} R={rc:.2f} F1={f1:.2f}")

# If diagnostic looks good (F1 jumps), proceed to full re-eval
print("\n" + "="*70)
print("If F1 above looks promising, re-run full eval with INCLUSIVE SF:")
print("="*70)

@torch.no_grad()
def final_eval_inclusive(data):
    model.eval()
    tp=fp=fn=0; rows=[]
    # also compute precision @ 1 (BioHopR-style sanity metric)
    p_at_1_hits = 0
    for i, ex in enumerate(data[:CFG['EVAL_N']]):
        prompt = build_prompt(ex['question'])
        enc = tok(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**enc, do_sample=False, max_new_tokens=CFG['MAX_NEW'],
                             pad_token_id=tok.pad_token_id)
        txt = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        sqs = parse_subqueries(txt)
        envout = env_execute(sqs, ex)
        pred = strict_format_inclusive(ex['question'], envout['pred_names'])
        P={p.lower() for p in pred}; G={g.lower() for g in ex['gold']}
        tp+=len(P&G); fp+=len(P-G); fn+=len(G-P)
        # BioHopR-style precision @ 1
        if pred and pred[0].lower() in G:
            p_at_1_hits += 1
        rows.append(dict(q=ex['question'], pred=pred, gold=ex['gold']))
        if (i+1)%100==0: print(f"  eval {i+1}/{CFG['EVAL_N']}")
    pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
    micro=2*pr*rc/(pr+rc) if pr+rc else 0
    p1 = p_at_1_hits/len(rows) if rows else 0
    json.dump(dict(micro_f1=micro, precision=pr, recall=rc,
                   precision_at_1=p1, n=len(rows)),
              open(CFG['OUT']+'/final_eval_inclusive.json','w'), indent=2)
    json.dump(rows, open(CFG['OUT']+'/final_preds_inclusive.json','w'), indent=2)
    print(f"\n=== BRACED (B11) INCLUSIVE: micro-F1={micro:.4f} "
          f"(P={pr:.3f} R={rc:.3f})  P@1={p1:.4f} (BioHopR-style) ===")
    return micro

# To run the full re-eval after the diagnostic:
# final_eval_inclusive(test)


In [ ]:
"""Compare 3 answer-head variants on 50 test (fast, mostly free).
Pick the winner for paper main result + report others as ablation.

Variant A: NO SF -- pred = top-K retrieval directly (free, fastest)
Variant B: INCLUSIVE SF (what we just tried)
Variant C: TOP-3 SF -- ask for top-3 most likely answers (compromise)
"""
import os, json, torch, re
from openai import OpenAI
oai = OpenAI()

PROMPT_INCLUSIVE = (
    "From the candidates below, output EVERY entity that correctly answers the "
    "question. The question may use singular phrasing but list ALL applicable. "
    "Use ONLY the provided candidates. Output comma-separated. If none, NONE.\n"
    "Question: {q}\nCandidates: {c}\nAll applicable answers:"
)
PROMPT_TOP3 = (
    "From the candidates below, output the TOP 3 most likely answers to the "
    "question, ranked by relevance. Use ONLY the provided candidates. "
    "Output comma-separated, 3 entities maximum.\n"
    "Question: {q}\nCandidates: {c}\nTop 3 answers:"
)
def sf_call(prompt, q, cand):
    r = oai.chat.completions.create(model='gpt-4o-mini', temperature=0, max_tokens=400,
        messages=[{'role':'user','content':prompt.format(q=q, c=', '.join(cand))}])
    txt = r.choices[0].message.content.strip()
    if not txt or txt.upper().startswith('NONE'): return []
    return [x.strip() for x in re.split(r'[,;\n]', txt) if x.strip()]

def f1_set(P, G):
    P={p.lower() for p in P}; G={g.lower() for g in G}
    if not P and not G: return 1.0,0,0,0
    if not P or not G:  return 0.0,0,0,len(G) if not P else len(P)
    tp=len(P&G); fp=len(P-G); fn=len(G-P)
    pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
    f1=2*pr*rc/(pr+rc) if pr+rc else 0
    return f1, tp, fp, fn

N = 50   # compare on 50 test for speed
model.eval()
res = {'no_sf':[], 'inclusive':[], 'top3':[]}
counts = {'no_sf':[0,0,0], 'inclusive':[0,0,0], 'top3':[0,0,0]}   # [tp,fp,fn]

print(f"Comparing 3 variants on first {N} test...")
for i in range(N):
    ex = test[i]
    prompt = build_prompt(ex['question'])
    enc = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, do_sample=False, max_new_tokens=CFG['MAX_NEW'],
                             pad_token_id=tok.pad_token_id)
    txt = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    sqs = parse_subqueries(txt)
    envout = env_execute(sqs, ex)
    cand = envout['pred_names']
    gold = ex['gold']

    # Variant A: no SF
    f1a, tpa, fpa, fna = f1_set(cand, gold)
    res['no_sf'].append(f1a)
    counts['no_sf'][0]+=tpa; counts['no_sf'][1]+=fpa; counts['no_sf'][2]+=fna

    # Variant B: inclusive SF
    pred_b = sf_call(PROMPT_INCLUSIVE, ex['question'], cand) if cand else []
    f1b, tpb, fpb, fnb = f1_set(pred_b, gold)
    res['inclusive'].append(f1b)
    counts['inclusive'][0]+=tpb; counts['inclusive'][1]+=fpb; counts['inclusive'][2]+=fnb

    # Variant C: top-3 SF
    pred_c = sf_call(PROMPT_TOP3, ex['question'], cand) if cand else []
    f1c, tpc, fpc, fnc = f1_set(pred_c, gold)
    res['top3'].append(f1c)
    counts['top3'][0]+=tpc; counts['top3'][1]+=fpc; counts['top3'][2]+=fnc

    if (i+1)%10==0: print(f"  {i+1}/{N} done")

import numpy as np
print("\n"+"="*70)
print(f"{'Variant':<14}{'micro-F1':>10}{'macro-F1':>10}{'P':>8}{'R':>8}")
print("="*70)
for v in ['no_sf','inclusive','top3']:
    tp,fp,fn = counts[v]
    pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
    micro=2*pr*rc/(pr+rc) if pr+rc else 0
    macro=float(np.mean(res[v]))
    print(f"{v:<14}{micro:>10.4f}{macro:>10.4f}{pr:>8.3f}{rc:>8.3f}")

print("\nInterpretation:")
print("  no_sf high   = retrieval is the bottleneck, SF adds noise")
print("  inclusive high = SF helps filter")
print("  top3 best    = constrained output reduces SF caution")
print("\nPick the winner for full 764 re-eval.")


In [ ]:
"""Full no-SF eval on all 764 test. FREE (no API), ~3-5 min.
Reports micro/macro F1, P, R, plus per-question breakdown for ablation."""
import os, json, torch, numpy as np

@torch.no_grad()
def full_eval_no_sf(data):
    model.eval()
    tp_g=fp_g=fn_g=0; per_q=[]
    sizes = []                          # gold size per question for analysis
    for i, ex in enumerate(data[:CFG['EVAL_N']]):
        prompt = build_prompt(ex['question'])
        enc = tok(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**enc, do_sample=False, max_new_tokens=CFG['MAX_NEW'],
                             pad_token_id=tok.pad_token_id)
        txt = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        sqs = parse_subqueries(txt)
        envout = env_execute(sqs, ex)
        pred = envout['pred_names']     # NO SF: candidates ARE the prediction
        P={p.lower() for p in pred}; G={g.lower() for g in ex['gold']}
        tp=len(P&G); fp=len(P-G); fn=len(G-P)
        tp_g+=tp; fp_g+=fp; fn_g+=fn
        pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
        f1=2*pr*rc/(pr+rc) if pr+rc else 0
        per_q.append(dict(f1=f1, p=pr, r=rc, n_pred=len(pred), n_gold=len(ex['gold'])))
        sizes.append(len(ex['gold']))
        if (i+1)%100==0: print(f"  eval {i+1}/{CFG['EVAL_N']}")

    pr=tp_g/(tp_g+fp_g) if tp_g+fp_g else 0
    rc=tp_g/(tp_g+fn_g) if tp_g+fn_g else 0
    micro=2*pr*rc/(pr+rc) if pr+rc else 0
    macro=float(np.mean([q['f1'] for q in per_q]))

    # bucket analysis: F1 by gold-size bucket
    buckets = {'≤5':[], '6-15':[], '16-50':[], '>50':[]}
    for q in per_q:
        n=q['n_gold']
        if n<=5: buckets['≤5'].append(q['f1'])
        elif n<=15: buckets['6-15'].append(q['f1'])
        elif n<=50: buckets['16-50'].append(q['f1'])
        else: buckets['>50'].append(q['f1'])

    result = dict(micro_f1=micro, macro_f1=macro, precision=pr, recall=rc,
                  n=len(per_q),
                  bucket_f1={k:(float(np.mean(v)) if v else 0.0, len(v)) for k,v in buckets.items()})
    json.dump(result, open(CFG['OUT']+'/final_eval_no_sf.json','w'), indent=2)
    json.dump(per_q, open(CFG['OUT']+'/per_q_no_sf.json','w'), indent=2)

    print("\n" + "="*70)
    print(f"=== BRACED (B11) FINAL no-SF: ===")
    print(f"  micro-F1 = {micro:.4f}  (P={pr:.3f}  R={rc:.3f})")
    print(f"  macro-F1 = {macro:.4f}")
    print(f"\n  F1 by gold-size bucket (Top-K=25 ceiling effect):")
    for k,(f,n) in result['bucket_f1'].items():
        print(f"    {k:<8}  n={n:>4}   F1={f:.4f}")
    print("="*70)
    return result

result = full_eval_no_sf(test)
